# Attention: every position reads every other one

MichAl Academy, unit 4.2.

Unit 3.10 left a recurrent network at chance on a task that only asks it to
remember the first token. This notebook runs the same task, on the same data,
with the same training budget, and replaces recurrence with one attention layer.

Nothing else is added: no feed-forward block, no residual connection, no layer
normalisation, no positional encoding. Those arrive in unit 4.3. What is measured
here is attention on its own.


In [ ]:
import time
import warnings

import numpy as np
import torch
from torch import nn

warnings.filterwarnings("ignore")
torch.set_num_threads(1)

CLASSES, FILLER = 8, 10          # tokens 1-8 are labels, 9-18 are filler
VOCAB = CLASSES + 1 + FILLER


def recall_task(n, length, seed):
    """First token is one of eight, the rest is noise, the label is that token."""
    rng = np.random.default_rng(seed)
    X = rng.integers(CLASSES + 1, CLASSES + 1 + FILLER,
                     size=(n, length)).astype(np.int64)
    y = rng.integers(0, CLASSES, n)
    X[:, 0] = y + 1
    return X, y


def train(model, X, y, epochs, lr=0.01, batch_size=64, seed=0):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    Xt, yt = torch.tensor(X), torch.tensor(y)
    for _ in range(epochs):
        order = torch.randperm(len(Xt))
        for i in range(0, len(Xt), batch_size):
            idx = order[i:i + batch_size]
            opt.zero_grad()
            loss_fn(model(Xt[idx]), yt[idx]).backward()
            opt.step()
    return model


def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        pred = model(torch.tensor(X)).argmax(dim=1)
    model.train()
    return float((pred == torch.tensor(y)).float().mean())


X, y = recall_task(4, 8, 0)
print("four sequences of length 8:")
print(X)
print("labels:", y)
print("\nThe label is the first token minus one. Everything from column 1 on is filler.")


## The arithmetic, on four vectors

Attention turns every position into three vectors. The **query** is what that
position is looking for, the **key** is what it offers to anyone looking, and the
**value** is what gets handed over when a query and a key match.

The score between position i and position j is the dot product of query i and
key j, divided by the square root of the vector length. Softmax over j turns that
row of scores into weights that sum to one, and the output at position i is the
weighted sum of the values.

This cell does it once, by hand, on four positions and two dimensions, with no
network and no training.


In [ ]:
q = torch.tensor([[1.0, 0.0]])                       # one query
k = torch.tensor([[1.0, 0.0],                        # four keys
                  [0.9, 0.4],
                  [0.0, 1.0],
                  [-1.0, 0.0]])
v = torch.tensor([[10.0], [20.0], [30.0], [40.0]])   # four values

scores = (q @ k.T) / (k.shape[1] ** 0.5)
weights = scores.softmax(dim=-1)

print("scores :", np.round(scores.numpy()[0], 4))
print("weights:", np.round(weights.numpy()[0], 4), " sum", float(weights.sum()))
print("output :", float(weights @ v))


The query points along the first dimension. Key 0 points the same way and scores
highest, key 3 points the opposite way and scores lowest, and key 2 is
perpendicular and scores zero. The output is nearer to value 10 than to anything
else, but it is a blend of all four, because softmax never returns a zero weight.

Two properties to hold on to:

- **The weights are a distribution.** They are positive and they sum to one, so
  attention always spends exactly one unit of attention per position.
- **Distance plays no part.** Position 3 is not harder to reach than position 1.
  A dot product does not know how far apart two positions are.

Distance is what the next cells measure.


## One attention layer

The model below embeds the tokens, computes queries, keys and values with three
linear layers, attends, and reads the answer off the last position. The readout
is the same one the recurrent networks of unit 3.10 used, so the comparison is
about how the information travels and nothing else.


In [ ]:
class SelfAttention(nn.Module):
    """Embedding, one multi-head self-attention layer, linear readout."""

    def __init__(self, vocab, classes, d=16, heads=1):
        super().__init__()
        self.heads = heads
        self.emb = nn.Embedding(vocab, d)
        self.q = nn.Linear(d, d)
        self.k = nn.Linear(d, d)
        self.v = nn.Linear(d, d)
        self.out = nn.Linear(d, classes)

    def weights(self, x):
        e = self.emb(x)
        B, L, d = e.shape
        dh = d // self.heads
        shape = (B, L, self.heads, dh)
        q = self.q(e).view(shape).transpose(1, 2)      # (B, heads, L, dh)
        k = self.k(e).view(shape).transpose(1, 2)
        return (q @ k.transpose(-2, -1)) / dh ** 0.5, self.v(e).view(shape).transpose(1, 2)

    def forward(self, x):
        scores, v = self.weights(x)
        ctx = (scores.softmax(dim=-1) @ v)             # (B, heads, L, dh)
        B, _, L, _ = ctx.shape
        ctx = ctx.transpose(1, 2).reshape(B, L, -1)    # heads back together
        return self.out(ctx[:, -1, :])


class Recurrent(nn.Module):
    """Unit 3.10's model, unchanged, so the two are trained on the same terms."""

    def __init__(self, kind, vocab, classes, hidden=32):
        super().__init__()
        self.emb = nn.Embedding(vocab, 16)
        self.rnn = (nn.RNN if kind == "rnn" else nn.LSTM)(16, hidden, batch_first=True)
        self.out = nn.Linear(hidden, classes)

    def forward(self, x):
        out, _ = self.rnn(self.emb(x))
        return self.out(out[:, -1, :])


def n_params(model):
    return sum(p.numel() for p in model.parameters())


print("attention:", n_params(SelfAttention(VOCAB, CLASSES)), "parameters")
print("plain RNN:", n_params(Recurrent("rnn", VOCAB, CLASSES)), "parameters")
print("LSTM     :", n_params(Recurrent("lstm", VOCAB, CLASSES)), "parameters")


The attention model is the smaller of the three. Whatever happens next is not
bought with extra capacity.

## The measurement

Same task, same six thousand training sequences, same forty epochs, same
optimiser and learning rate. Chance is 0.125.


In [ ]:
started = time.time()
print("length   attention   plain RNN   LSTM")
rows = {}
for length in (40, 60):
    X_tr, y_tr = recall_task(6000, length, 30 + length)
    X_te, y_te = recall_task(1000, length, 40 + length)
    row = []
    for build in (lambda: SelfAttention(VOCAB, CLASSES),
                  lambda: Recurrent("rnn", VOCAB, CLASSES),
                  lambda: Recurrent("lstm", VOCAB, CLASSES)):
        torch.manual_seed(0)
        model = train(build(), X_tr, y_tr, epochs=40)
        row.append(accuracy(model, X_te, y_te))
    rows[length] = row
    print(f"{length:6d}   {row[0]:.4f}      {row[1]:.4f}      {row[2]:.4f}")
print(f"({time.time() - started:.1f}s)")


## Where the attention went

An accuracy is a result. The weights are the explanation, and they can be read
directly: for every position, they say which position it took its information
from.


In [ ]:
torch.manual_seed(0)
length = 60
X_tr, y_tr = recall_task(6000, length, 30 + length)
X_te, y_te = recall_task(1000, length, 40 + length)
model = train(SelfAttention(VOCAB, CLASSES), X_tr, y_tr, epochs=40)

model.eval()
with torch.no_grad():
    scores, _ = model.weights(torch.tensor(X_te[:256]))
    w = scores.softmax(dim=-1)[:, 0]        # (256, 60, 60), single head

on_first = float(w[:, :, 0].mean())
print(f"average weight on position 0, over all positions: {on_first:.4f}")
print(f"a uniform model would give it:                    {1 / length:.4f}")
print(f"weight from position 30 to position 0:            {float(w[:, 30, 0].mean()):.4f}")
print(f"weight from the last position to position 0:      {float(w[:, -1, 0].mean()):.4f}")


Nearly all of it lands on position 0, from everywhere. The last position reaches
the first one in a single step, and the recurrent networks had to carry it
through sixty multiplications to do the same job.

Position 0 holds the only token in the sequence drawn from the label vocabulary,
so the model is finding it **by content, not by position**. Nothing in this layer
knows what order the tokens are in; unit 4.3 is where that gets added, and it has
to be added on purpose.

## Does adding heads help

Multi-head attention splits the vectors into groups and runs the same operation
on each group separately, so several positions can be attended to for different
reasons at once. The total width is held fixed, so more heads means narrower
ones.


In [ ]:
started = time.time()
X_tr, y_tr = recall_task(6000, 60, 90)
X_te, y_te = recall_task(1000, 60, 91)
for heads in (1, 2, 4):
    torch.manual_seed(0)
    model = train(SelfAttention(VOCAB, CLASSES, heads=heads), X_tr, y_tr, epochs=40)
    print(f"{heads} head(s), width {16 // heads} each: {accuracy(model, X_te, y_te):.4f}")
print(f"({time.time() - started:.1f}s)")


No difference. This task has exactly one thing worth looking at, and one head is
enough to look at it, so the extra heads have nothing to do.

That is the honest reading of multi-head attention: it is capacity for attending
to several things at once, not a free accuracy increase. A task that needs one
relationship gets nothing from it.

## What this notebook did not show

**Cost.** Every position scores against every other one, so the score matrix is
length by length. Doubling the sequence quadruples that matrix, while a recurrent
network doubles. At length 60 nobody notices; at length 60,000 it is the reason
long contexts are expensive.

**Order.** The model above cannot tell `dog bites man` from `man bites dog`,
because a set of tokens goes in and a set of tokens comes out. It solved the
recall task because the answer happened to be the odd token out rather than the
first one.
